In [ ]:
!pip install rioxarray geopandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis


In [ ]:
# =============================================================
# CMIP6 Station-wise Extraction for Multiple Models & Scenarios
# =============================================================

import os
import xarray as xr
import rioxarray
import geopandas as gpd
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# =============================================================
# USER INPUTS
# =============================================================

# Root directory containing all CMIP6 models
BASE_PATH = "/content/drive/MyDrive/NEX_GDDP_CMIP6"

# Variable name
VARIABLE = "pr"

# Scenarios to process
SCENARIOS = [
    "historical",
    "ssp245",
    "ssp585"
]

# Basin shapefile
SHP_PATH = "Basin_shp/Karnali_basin.shp"

# Station file
STATION_FILE = "Stationwise/Karnali_1986_2015.xlsx"

# Output directory
OUTPUT_ROOT = "Stationwise/Precipitation/Raw_GCM_Precipitation"

# =============================================================
# READ BASIN
# =============================================================

print("Reading basin...")

basin = gpd.read_file(SHP_PATH)

if basin.crs != "EPSG:4326":
    basin = basin.to_crs("EPSG:4326")

# =============================================================
# READ STATIONS
# =============================================================

print("Reading stations...")

stations = pd.read_excel(
    STATION_FILE,
    sheet_name="Sheet1"
)

station_ids = stations["Index No."].astype(str).values

station_lats = xr.DataArray(
    stations["Lat"].values,
    dims="station",
    coords={"station": station_ids}
)

station_lons = xr.DataArray(
    stations["Lon"].values,
    dims="station",
    coords={"station": station_ids}
)

# =============================================================
# FIND AVAILABLE MODELS
# =============================================================

models = sorted([
    d for d in os.listdir(BASE_PATH)
    if os.path.isdir(os.path.join(BASE_PATH, d))
])

print(models)

models = ['ACCESS-CM2', 'ACCESS-ESM1-5']


Reading basin...
Reading stations...
['ACCESS-CM2', 'ACCESS-ESM1-5', 'BCC-CSM2-MR', 'CanESM5', 'EC-Earth3', 'EC-Earth3-Veg', 'GFDL-ESM4', 'INM-CM4-8', 'INM-CM5-0', 'IPSL-CM6A-LR', 'MIROC6', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'clipped_gcm']


In [ ]:
# =============================================================
# MAIN LOOP
# =============================================================

for GCM in models:

    print("\n")
    print("=" * 70)
    print("Processing Model :", GCM)
    print("=" * 70)

    for scenario in SCENARIOS:

        print(f"\nScenario : {scenario}")

        data_path = os.path.join(
            BASE_PATH,
            GCM,
            scenario,
            VARIABLE
        )

        # ---------------------------------------------------------
        # Check folder
        # ---------------------------------------------------------

        if not os.path.exists(data_path):
            print("Folder not found.")
            continue

        files = sorted([
            os.path.join(data_path, f)
            for f in os.listdir(data_path)
            if f.endswith(".nc")
        ])

        if len(files) == 0:
            print("No NetCDF files.")
            continue

        print(f"NetCDF files : {len(files)}")

        try:

            # =====================================================
            # READ DATA
            # =====================================================

            ds = xr.open_mfdataset(
                files,
                combine="by_coords",
                parallel=True
            )

            ds = ds.rio.write_crs("EPSG:4326")

            # =====================================================
            # CLIP BASIN
            # =====================================================

            ds_clip = ds.rio.clip(
                basin.geometry,
                basin.crs,
                drop=True
            )

            pr = ds_clip[VARIABLE]

            # =====================================================
            # EXTRACT STATIONS
            # =====================================================

            station_pr = pr.sel(
                lat=station_lats,
                lon=station_lons,
                method="nearest"
            )

            # =====================================================
            # Convert kg/m2/s → mm/day
            # =====================================================

            station_pr = station_pr * 86400

            # =====================================================
            # CREATE DATAFRAME
            # =====================================================

            df = pd.DataFrame(
                station_pr.values,
                index=station_pr.time.values,
                columns=station_ids
            )

            df.index.name = "Date"

            df.reset_index(inplace=True)

            df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()

            # =====================================================
            # Rename columns with selected grid coordinates
            # =====================================================

            new_columns = {"Date": "Date"}

            for i, sid in enumerate(station_ids):

                grid_lat = float(station_pr.lat.values[i])
                grid_lon = float(station_pr.lon.values[i])

                new_columns[sid] = (
                    f"{sid} ({grid_lat:.3f}, {grid_lon:.3f})"
                )

            df.rename(columns=new_columns, inplace=True)

            # =====================================================
            # SAVE
            # =====================================================

            out_dir = os.path.join(
                OUTPUT_ROOT,
                GCM,
                scenario
            )

            os.makedirs(out_dir, exist_ok=True)

            out_csv = os.path.join(
                out_dir,
                f"{GCM}_{scenario}_karnali_pr.csv"
            )

            df.to_csv(out_csv, index=False)

            print("Saved :", out_csv)
            print("Stations :", len(station_ids))
            print("Days :", len(df))

            ds.close()

        except Exception as e:

            print(f"Failed : {GCM} {scenario}")
            print(e)

# =============================================================
# FINISHED
# =============================================================

print("\n")
print("=" * 70)
print("ALL PROCESSING COMPLETED")
print("=" * 70)



Processing Model : ACCESS-CM2

Scenario : historical
NetCDF files : 30
Saved : Stationwise/Precipitation/Raw_GCM_Precipitation/ACCESS-CM2/historical/ACCESS-CM2_historical_karnali_pr.csv
Stations : 10
Days : 10957

Scenario : ssp245
NetCDF files : 86
